# 09 — Final model-family benchmark

This notebook freezes and compares the six operational model families used in the manuscript:

1. **Baseline**
2. **Statistical**
3. **Machine learning**
4. **Deep learning**
5. **Advanced traditional**
6. **Robust hybrid**

The benchmark contains 40 forecasting tasks: five temporal resolutions × two targets × four forecast horizons. Configuration choices are made from validation evidence only. Test predictions are read once after those choices have been frozen.

The notebook also enforces exact forecast-origin parity across families, verifies the `BASE_ONLY` equivalence between the robust strategy and the selected advanced-traditional prediction, recomputes all metrics from prediction-level data, and produces the English source table for Table 1.

## Scientific contract

- Run notebooks `01`–`08` first in the same project directory.
- No model or configuration may be selected using test performance.
- Baseline and statistical representatives are selected per forecasting task from validation RMSE.
- Machine-learning configurations are imported from notebook `05`'s validation-only selection.
- Deep-learning architecture is selected per task from validation ensemble RMSE; architectures within 1% use seed stability and a fixed architecture priority as tie-breakers.
- Advanced-traditional predictions are imported from notebook `07` after its resolution × target validation selection.
- Robust-hybrid predictions are imported from notebook `08` after its task-level multi-window selector.
- The advanced-traditional output is the canonical source for target values and test-origin keys.
- NRMSE is recomputed as test RMSE divided by the task-specific standard deviation of observed test targets, matching the corrected historical benchmark.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

In [ ]:
def locate_project_root():
    explicit = os.environ.get("GREENHOUSE_PROJECT_ROOT")
    if explicit:
        root = Path(explicit).expanduser().resolve()
        if not (root / "results").exists():
            raise FileNotFoundError(f"GREENHOUSE_PROJECT_ROOT has no results directory: {root}")
        return root
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Set GREENHOUSE_PROJECT_ROOT or run from the repository."
    )


PROJECT_ROOT = locate_project_root()
RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_DIR = RESULTS_ROOT / "final_benchmark"
PREDICTION_DIR = RESULTS_DIR / "predictions"
METADATA_DIR = PROJECT_ROOT / "metadata"
for directory in [RESULTS_DIR, PREDICTION_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

EXECUTION_MODE = os.environ.get("GREENHOUSE_EXECUTION_MODE", "full").strip().lower()
if EXECUTION_MODE not in {"full", "smoke_test"}:
    raise ValueError("GREENHOUSE_EXECUTION_MODE must be 'full' or 'smoke_test'.")

RESOLUTIONS = [4, 12, 20, 30, 60]
ACTIVE_RESOLUTIONS = RESOLUTIONS if EXECUTION_MODE == "full" else [60]
TARGETS = ["temperature", "relative_humidity"]
HORIZONS = [60, 120, 240, 480]
FAMILIES = [
    "BASELINE", "STATISTICAL", "MACHINE_LEARNING",
    "DEEP_LEARNING", "ADVANCED_TRADITIONAL", "HYBRID_ROBUST",
]
FAMILY_LABELS = {
    "BASELINE": "Baseline",
    "STATISTICAL": "Statistical",
    "MACHINE_LEARNING": "Machine learning",
    "DEEP_LEARNING": "Deep learning",
    "ADVANCED_TRADITIONAL": "Advanced traditional",
    "HYBRID_ROBUST": "Robust hybrid",
}
TASK_COLUMNS = ["resolution_minutes", "target", "horizon_minutes"]
KEY_COLUMNS = TASK_COLUMNS + ["origin_index"]
EXPECTED_TASKS = len(ACTIVE_RESOLUTIONS) * len(TARGETS) * len(HORIZONS)
TRUTH_ATOL = 1e-3
TRUTH_RTOL = 1e-6

print({
    "project_root": str(PROJECT_ROOT),
    "execution_mode": EXECUTION_MODE,
    "active_resolutions": ACTIVE_RESOLUTIONS,
    "expected_tasks_per_family": EXPECTED_TASKS,
})

## Input interfaces and schema audit

In [ ]:
INPUTS = {
    "baseline_statistical_predictions": RESULTS_ROOT / "baselines_statistical" / "07_all_predictions.csv",
    "machine_learning_predictions": RESULTS_ROOT / "machine_learning" / "09_ml_predictions.csv",
    "machine_learning_selection": RESULTS_ROOT / "machine_learning" / "06_selected_feature_set_by_resolution.csv",
    "deep_learning_predictions": RESULTS_ROOT / "deep_learning" / "10_dl_predictions.csv",
    "deep_learning_architecture_summary": RESULTS_ROOT / "deep_learning" / "05_architecture_validation_summary.csv",
    "deep_learning_seed_metrics": RESULTS_ROOT / "deep_learning" / "03_seed_validation_metrics.csv",
    "advanced_test_predictions": RESULTS_ROOT / "advanced_traditional" / "predictions" / "02_selected_advanced_traditional_test_predictions.parquet",
    "advanced_selection": RESULTS_ROOT / "advanced_traditional" / "07_selected_configuration_by_resolution_target.csv",
    "robust_test_predictions": RESULTS_ROOT / "robust_residual_hybrid" / "predictions" / "06_robust_test_predictions.parquet",
    "robust_selection": RESULTS_ROOT / "robust_residual_hybrid" / "10_robust_operational_selection.csv",
}

missing = [str(path) for path in INPUTS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Run notebooks 04–08 first. Missing:\n" + "\n".join(missing))


def read_table(path):
    return pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)


def require_columns(frame, required, name):
    absent = sorted(set(required) - set(frame.columns))
    if absent:
        raise ValueError(f"{name} is missing required columns: {absent}")


input_audit_rows = []
raw = {}
for name, path in INPUTS.items():
    frame = read_table(path)
    raw[name] = frame
    input_audit_rows.append({
        "input": name,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "rows": len(frame),
        "columns": len(frame.columns),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })

input_audit = pd.DataFrame(input_audit_rows)
input_audit.to_csv(RESULTS_DIR / "01_input_interface_audit.csv", index=False)
display(input_audit)

In [ ]:
prediction_requirements = [
    "resolution_minutes", "target", "horizon_minutes", "origin_index", "observed", "predicted"
]
require_columns(raw["baseline_statistical_predictions"], prediction_requirements + ["model", "family", "split"], "notebook 04 predictions")
require_columns(raw["machine_learning_predictions"], prediction_requirements + ["model", "feature_set", "candidate_id", "split"], "notebook 05 predictions")
require_columns(raw["machine_learning_selection"], ["resolution_minutes", "model", "feature_set", "candidate_id"], "notebook 05 selection")
require_columns(raw["deep_learning_predictions"], prediction_requirements + ["architecture", "split"], "notebook 06 predictions")
require_columns(
    raw["deep_learning_architecture_summary"],
    ["resolution_minutes", "architecture", "candidate_id"],
    "notebook 06 architecture summary",
)
require_columns(raw["deep_learning_seed_metrics"], TASK_COLUMNS + ["architecture", "candidate_id", "seed", "rmse"], "notebook 06 seed metrics")
require_columns(raw["advanced_test_predictions"], prediction_requirements + ["family", "candidate_id", "split"], "notebook 07 predictions")
require_columns(raw["advanced_selection"], ["resolution_minutes", "target", "family", "candidate_id"], "notebook 07 selection")
require_columns(raw["robust_test_predictions"], [
    "resolution_minutes", "target", "horizon_minutes", "origin_index", "observed",
    "robust_predicted", "operational_action", "operational_model",
], "notebook 08 predictions")
require_columns(raw["robust_selection"], TASK_COLUMNS + ["corrector", "alpha", "selection_reason"], "notebook 08 selection")

dl_candidate_map = raw["deep_learning_architecture_summary"][[
    "resolution_minutes", "architecture", "candidate_id"
]].copy()
dl_candidate_map["resolution_minutes"] = pd.to_numeric(
    dl_candidate_map["resolution_minutes"]
).astype(int)
dl_candidate_map["candidate_id"] = pd.to_numeric(dl_candidate_map["candidate_id"]).astype(int)
if dl_candidate_map.duplicated(["resolution_minutes", "architecture"]).any():
    raise ValueError(
        "Notebook 06 architecture summary contains more than one selected candidate "
        "for the same resolution and architecture."
    )

if "candidate_id" not in raw["deep_learning_predictions"].columns:
    raw["deep_learning_predictions"] = raw["deep_learning_predictions"].merge(
        dl_candidate_map,
        on=["resolution_minutes", "architecture"],
        how="left",
        validate="many_to_one",
    )
    if raw["deep_learning_predictions"]["candidate_id"].isna().any():
        missing_keys = (
            raw["deep_learning_predictions"]
            .loc[lambda frame: frame["candidate_id"].isna(), ["resolution_minutes", "architecture"]]
            .drop_duplicates()
            .to_dict("records")
        )
        raise ValueError(
            "Could not reconstruct candidate_id for notebook 06 predictions. "
            f"Missing architecture-summary keys: {missing_keys}"
        )
    deep_learning_candidate_id_source = (
        "reconstructed from notebook 06 architecture validation summary"
    )
else:
    deep_learning_candidate_id_source = "provided directly by notebook 06 predictions"

raw["deep_learning_predictions"]["candidate_id"] = pd.to_numeric(
    raw["deep_learning_predictions"]["candidate_id"]
).astype(int)
require_columns(
    raw["deep_learning_predictions"],
    prediction_requirements + ["architecture", "candidate_id", "split"],
    "notebook 06 predictions after interface reconciliation",
)
print(f"Deep-learning candidate_id: {deep_learning_candidate_id_source}")

## Freeze the family representatives

The following cells produce one prediction per family and forecast origin. Every selection table records its validation-only basis. Test metrics are not computed until all six representatives have been assembled.

In [ ]:
def prepare_predictions(frame):
    result = frame.copy()
    result["resolution_minutes"] = pd.to_numeric(result["resolution_minutes"]).astype(int)
    result["horizon_minutes"] = pd.to_numeric(result["horizon_minutes"]).astype(int)
    result["origin_index"] = pd.to_numeric(result["origin_index"]).astype(int)
    result["target"] = result["target"].astype(str).str.lower()
    return result.loc[
        result["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)
        & result["target"].isin(TARGETS)
        & result["horizon_minutes"].isin(HORIZONS)
    ].copy()


def validation_metrics_from_predictions(frame, group_columns):
    rows = []
    for keys, group in frame.groupby(group_columns, sort=False, observed=True):
        y = group["observed"].to_numpy(dtype=float)
        p = group["predicted"].to_numpy(dtype=float)
        row = dict(zip(group_columns, keys if isinstance(keys, tuple) else (keys,)))
        row.update({
            "validation_n": len(group),
            "validation_rmse": float(np.sqrt(mean_squared_error(y, p))),
            "validation_r2": float(r2_score(y, p)),
        })
        rows.append(row)
    return pd.DataFrame(rows)


def select_simple_family(source, source_family, output_family):
    family_data = prepare_predictions(source)
    family_data = family_data.loc[family_data["family"].astype(str).str.lower().eq(source_family)].copy()
    validation = family_data.loc[family_data["split"].eq("validation")]
    scores = validation_metrics_from_predictions(validation, TASK_COLUMNS + ["model"])
    winners = (
        scores.sort_values(TASK_COLUMNS + ["validation_rmse", "validation_r2", "model"],
                           ascending=[True, True, True, True, False, True])
        .groupby(TASK_COLUMNS, as_index=False, sort=False).first()
    )
    test = family_data.loc[family_data["split"].eq("test")].merge(
        winners[TASK_COLUMNS + ["model", "validation_rmse", "validation_r2"]],
        on=TASK_COLUMNS + ["model"], how="inner", validate="many_to_one"
    )
    test["family"] = output_family
    test["selected_configuration"] = test["model"]
    winners["family"] = output_family
    winners["selection_scope"] = "resolution × target × horizon"
    winners["selection_basis"] = "minimum validation RMSE; validation R2 then model name as tie-breakers"
    winners["test_used_for_selection"] = False
    return test, winners


baseline_test, baseline_selection = select_simple_family(
    raw["baseline_statistical_predictions"], "reference", "BASELINE"
)
statistical_test, statistical_selection = select_simple_family(
    raw["baseline_statistical_predictions"], "statistical", "STATISTICAL"
)

In [ ]:
ml_predictions = prepare_predictions(raw["machine_learning_predictions"])
ml_selection_source = raw["machine_learning_selection"].copy()
ml_selection_source["resolution_minutes"] = pd.to_numeric(ml_selection_source["resolution_minutes"]).astype(int)
ml_selection_source = ml_selection_source.loc[
    ml_selection_source["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)
].copy()

machine_learning_test = ml_predictions.loc[ml_predictions["split"].eq("test")].merge(
    ml_selection_source[["resolution_minutes", "model", "feature_set", "candidate_id"]],
    on=["resolution_minutes", "model", "feature_set", "candidate_id"],
    how="inner", validate="many_to_one",
)
machine_learning_test["family"] = "MACHINE_LEARNING"
machine_learning_test["selected_configuration"] = (
    machine_learning_test["model"].astype(str) + ":" + machine_learning_test["feature_set"].astype(str)
)
machine_learning_selection = ml_selection_source.copy()
machine_learning_selection["family"] = "MACHINE_LEARNING"
machine_learning_selection["selection_scope"] = "resolution"
machine_learning_selection["selection_basis"] = "notebook 05 validation-only algorithm and feature-set selection"
machine_learning_selection["test_used_for_selection"] = False

In [ ]:
dl_predictions = prepare_predictions(raw["deep_learning_predictions"])
dl_validation = dl_predictions.loc[dl_predictions["split"].eq("validation")].copy()
dl_test_all = dl_predictions.loc[dl_predictions["split"].eq("test")].copy()

dl_validation_scores = validation_metrics_from_predictions(
    dl_validation, TASK_COLUMNS + ["architecture", "candidate_id"]
)
seed_metrics = raw["deep_learning_seed_metrics"].copy()
seed_metrics["resolution_minutes"] = pd.to_numeric(seed_metrics["resolution_minutes"]).astype(int)
seed_metrics["horizon_minutes"] = pd.to_numeric(seed_metrics["horizon_minutes"]).astype(int)
seed_metrics = seed_metrics.loc[
    seed_metrics["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)
    & seed_metrics["target"].astype(str).str.lower().isin(TARGETS)
    & seed_metrics["horizon_minutes"].isin(HORIZONS)
].copy()
seed_stability = (
    seed_metrics.groupby(TASK_COLUMNS + ["architecture", "candidate_id"], as_index=False)
    .agg(validation_seed_rmse_sd=("rmse", "std"), independent_seeds=("seed", "nunique"))
)
seed_stability["validation_seed_rmse_sd"] = seed_stability["validation_seed_rmse_sd"].fillna(0.0)
dl_validation_scores = dl_validation_scores.merge(
    seed_stability, on=TASK_COLUMNS + ["architecture", "candidate_id"], how="left", validate="one_to_one"
)
dl_validation_scores["validation_seed_rmse_sd"] = dl_validation_scores["validation_seed_rmse_sd"].fillna(np.inf)
dl_validation_scores["minimum_validation_rmse"] = dl_validation_scores.groupby(TASK_COLUMNS)["validation_rmse"].transform("min")
dl_validation_scores["relative_gap_to_minimum"] = (
    dl_validation_scores["validation_rmse"] / dl_validation_scores["minimum_validation_rmse"] - 1.0
)
dl_validation_scores["within_one_percent"] = dl_validation_scores["relative_gap_to_minimum"] <= 0.01 + 1e-12
architecture_priority = {"MLP": 1, "GRU": 2, "LSTM": 3, "TCN": 4}
eligible_dl = dl_validation_scores.loc[dl_validation_scores["within_one_percent"]].copy()
eligible_dl["architecture_priority"] = eligible_dl["architecture"].map(architecture_priority).fillna(999)
deep_learning_selection = (
    eligible_dl.sort_values(
        TASK_COLUMNS + ["validation_seed_rmse_sd", "validation_rmse", "architecture_priority"],
        ascending=[True, True, True, True, True, True],
    ).groupby(TASK_COLUMNS, as_index=False, sort=False).first()
)
deep_learning_selection["family"] = "DEEP_LEARNING"
deep_learning_selection["selection_scope"] = "resolution × target × horizon"
deep_learning_selection["selection_basis"] = (
    "minimum validation ensemble RMSE; within 1%, seed RMSE SD and fixed architecture priority"
)
deep_learning_selection["test_used_for_selection"] = False

deep_learning_test = dl_test_all.merge(
    deep_learning_selection[TASK_COLUMNS + ["architecture", "candidate_id", "validation_rmse", "validation_seed_rmse_sd"]],
    on=TASK_COLUMNS + ["architecture", "candidate_id"], how="inner", validate="many_to_one"
)
deep_learning_test["family"] = "DEEP_LEARNING"
deep_learning_test["selected_configuration"] = deep_learning_test["architecture"]

In [ ]:
advanced_test = prepare_predictions(raw["advanced_test_predictions"])
advanced_test = advanced_test.loc[advanced_test["split"].eq("test")].copy()
advanced_test["family"] = "ADVANCED_TRADITIONAL"
advanced_test["selected_configuration"] = advanced_test["family_y"] if "family_y" in advanced_test else advanced_test.get("family", "ADVANCED_TRADITIONAL")
if "family" in raw["advanced_test_predictions"].columns:
    advanced_test["selected_configuration"] = prepare_predictions(raw["advanced_test_predictions"]).loc[
        lambda x: x["split"].eq("test"), "family"
    ].to_numpy()
advanced_selection = raw["advanced_selection"].copy()
advanced_selection["resolution_minutes"] = pd.to_numeric(advanced_selection["resolution_minutes"]).astype(int)
advanced_selection = advanced_selection.loc[advanced_selection["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)].copy()
advanced_selection["source_model"] = advanced_selection["family"]
advanced_selection["family"] = "ADVANCED_TRADITIONAL"
advanced_selection["selection_scope"] = "resolution × target"
advanced_selection["selection_basis"] = "notebook 07 mean validation NRMSE across horizons; validation R2 secondary"
advanced_selection["test_used_for_selection"] = False

robust_source = raw["robust_test_predictions"].copy()
robust_source["predicted"] = robust_source["robust_predicted"]
robust_test = prepare_predictions(robust_source)
robust_test["family"] = "HYBRID_ROBUST"
robust_test["selected_configuration"] = robust_test["operational_model"]
robust_selection = raw["robust_selection"].copy()
robust_selection["resolution_minutes"] = pd.to_numeric(robust_selection["resolution_minutes"]).astype(int)
robust_selection["horizon_minutes"] = pd.to_numeric(robust_selection["horizon_minutes"]).astype(int)
robust_selection = robust_selection.loc[robust_selection["resolution_minutes"].isin(ACTIVE_RESOLUTIONS)].copy()
robust_selection["family"] = "HYBRID_ROBUST"
robust_selection["selection_scope"] = "resolution × target × horizon"
robust_selection["selection_basis"] = "notebook 08 validation-only four-window robustness rule"
robust_selection["test_used_for_selection"] = False

## Assemble the frozen prediction panel and enforce origin parity

In [ ]:
COMMON_COLUMNS = KEY_COLUMNS + ["observed", "predicted", "family", "selected_configuration"]
family_frames = {
    "BASELINE": baseline_test,
    "STATISTICAL": statistical_test,
    "MACHINE_LEARNING": machine_learning_test,
    "DEEP_LEARNING": deep_learning_test,
    "ADVANCED_TRADITIONAL": advanced_test,
    "HYBRID_ROBUST": robust_test,
}

prediction_frames = []
for family, frame in family_frames.items():
    prepared = frame.copy()
    prepared["family"] = family
    require_columns(prepared, COMMON_COLUMNS, family)
    if prepared.duplicated(KEY_COLUMNS).any():
        examples = prepared.loc[prepared.duplicated(KEY_COLUMNS, keep=False), KEY_COLUMNS].head()
        raise AssertionError(f"Duplicate prediction keys for {family}:\n{examples}")
    prediction_frames.append(prepared[COMMON_COLUMNS])

final_predictions = pd.concat(prediction_frames, ignore_index=True)
canonical = final_predictions.loc[final_predictions["family"].eq("ADVANCED_TRADITIONAL")].copy()
canonical_keys = canonical[KEY_COLUMNS].sort_values(KEY_COLUMNS).reset_index(drop=True)

parity_rows = []
for family in FAMILIES:
    subset = final_predictions.loc[final_predictions["family"].eq(family)].copy()
    keys = subset[KEY_COLUMNS].sort_values(KEY_COLUMNS).reset_index(drop=True)
    key_match = keys.equals(canonical_keys)
    merged = canonical[KEY_COLUMNS + ["observed"]].merge(
        subset[KEY_COLUMNS + ["observed"]], on=KEY_COLUMNS, how="outer",
        suffixes=("_canonical", "_family"), indicator=True,
    )
    truth_match = bool(
        (merged["_merge"] == "both").all()
        and np.allclose(
            merged["observed_canonical"], merged["observed_family"],
            atol=TRUTH_ATOL, rtol=TRUTH_RTOL, equal_nan=False,
        )
    )
    parity_rows.append({
        "family": family,
        "prediction_rows": len(subset),
        "tasks": subset[TASK_COLUMNS].drop_duplicates().shape[0],
        "key_match_to_canonical": key_match,
        "truth_match_to_canonical": truth_match,
        "missing_or_extra_keys": int((merged["_merge"] != "both").sum()),
        "maximum_truth_absolute_difference": float(
            np.nanmax(np.abs(merged["observed_canonical"] - merged["observed_family"]))
        ) if (merged["_merge"] == "both").any() else np.nan,
    })

origin_parity_audit = pd.DataFrame(parity_rows)
assert origin_parity_audit["key_match_to_canonical"].all(), origin_parity_audit
assert origin_parity_audit["truth_match_to_canonical"].all(), origin_parity_audit
assert (origin_parity_audit["tasks"] == EXPECTED_TASKS).all(), origin_parity_audit
origin_parity_audit.to_csv(RESULTS_DIR / "03_origin_and_truth_parity_audit.csv", index=False)
display(origin_parity_audit)

In [ ]:
# Canonicalize observed values after the parity check.
final_predictions = final_predictions.drop(columns="observed").merge(
    canonical[KEY_COLUMNS + ["observed"]], on=KEY_COLUMNS, how="left", validate="many_to_one"
)

# BASE_ONLY must exactly reproduce the advanced-traditional operational forecast.
robust_detail = robust_test[KEY_COLUMNS + ["operational_action", "predicted"]].rename(
    columns={"predicted": "hybrid_predicted"}
)
advanced_detail = advanced_test[KEY_COLUMNS + ["predicted"]].rename(
    columns={"predicted": "advanced_predicted"}
)
base_only_audit = robust_detail.loc[robust_detail["operational_action"].eq("BASE_ONLY")].merge(
    advanced_detail, on=KEY_COLUMNS, how="left", validate="one_to_one"
)
base_only_audit["absolute_difference"] = np.abs(
    base_only_audit["hybrid_predicted"] - base_only_audit["advanced_predicted"]
)
if len(base_only_audit):
    assert np.allclose(
        base_only_audit["hybrid_predicted"], base_only_audit["advanced_predicted"],
        atol=1e-12, rtol=0.0,
    ), base_only_audit.sort_values("absolute_difference", ascending=False).head()
base_only_summary = pd.DataFrame([{
    "base_only_prediction_rows": len(base_only_audit),
    "base_only_tasks": base_only_audit[TASK_COLUMNS].drop_duplicates().shape[0],
    "maximum_absolute_difference": float(base_only_audit["absolute_difference"].max()) if len(base_only_audit) else 0.0,
    "exact_equivalence": bool((base_only_audit["absolute_difference"] <= 1e-12).all()) if len(base_only_audit) else True,
}])
base_only_summary.to_csv(RESULTS_DIR / "04_base_only_exact_equivalence_audit.csv", index=False)
display(base_only_summary)

## Recompute task-level metrics from the frozen predictions

In [ ]:
metric_rows = []
for keys, group in final_predictions.groupby(["family"] + TASK_COLUMNS, sort=False, observed=True):
    y = group["observed"].to_numpy(dtype=float)
    p = group["predicted"].to_numpy(dtype=float)
    scale = float(np.std(y, ddof=0))
    rmse = float(np.sqrt(mean_squared_error(y, p)))
    metric_rows.append({
        "family": keys[0],
        "resolution_minutes": int(keys[1]),
        "target": keys[2],
        "horizon_minutes": int(keys[3]),
        "n_origins": len(group),
        "rmse": rmse,
        "nrmse": rmse / scale if scale > 0 else np.nan,
        "mae": float(mean_absolute_error(y, p)),
        "bias": float(np.mean(p - y)),
        "r2": float(r2_score(y, p)),
        "test_target_sd": scale,
    })

task_metrics = pd.DataFrame(metric_rows)
assert len(task_metrics) == len(FAMILIES) * EXPECTED_TASKS
assert task_metrics[["rmse", "nrmse", "mae", "r2"]].apply(np.isfinite).all().all()
task_metrics["rmse_rank"] = task_metrics.groupby(TASK_COLUMNS)["rmse"].rank(method="average")
task_metrics["r2_rank"] = task_metrics.groupby(TASK_COLUMNS)["r2"].rank(method="average", ascending=False)

family_task_audit = (
    task_metrics.groupby("family", as_index=False)
    .agg(tasks=("rmse", "size"), finite_rmse=("rmse", lambda s: int(np.isfinite(s).sum())),
         minimum_origins=("n_origins", "min"), maximum_origins=("n_origins", "max"))
)
task_metrics.to_csv(RESULTS_DIR / "05_final_family_task_metrics.csv", index=False)
family_task_audit.to_csv(RESULTS_DIR / "06_family_task_audit.csv", index=False)
display(family_task_audit)

In [ ]:
aggregate_performance = (
    task_metrics.groupby(["family", "target"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_r2=("r2", "mean"),
        mean_nrmse=("nrmse", "mean"),
        median_nrmse=("nrmse", "median"),
        mean_rmse_rank=("rmse_rank", "mean"),
        tasks=("rmse", "size"),
    )
)
aggregate_performance["family_label"] = aggregate_performance["family"].map(FAMILY_LABELS)
aggregate_performance.to_csv(RESULTS_DIR / "07_aggregate_family_performance.csv", index=False)

ranking = (
    task_metrics.groupby("family", as_index=False)
    .agg(
        average_rmse_rank=("rmse_rank", "mean"),
        median_rmse_rank=("rmse_rank", "median"),
        average_r2_rank=("r2_rank", "mean"),
        mean_nrmse=("nrmse", "mean"),
        median_nrmse=("nrmse", "median"),
        mean_r2=("r2", "mean"),
    )
    .sort_values("average_rmse_rank")
)
ranking["family_label"] = ranking["family"].map(FAMILY_LABELS)
ranking.to_csv(RESULTS_DIR / "08_final_family_ranking.csv", index=False)
display(aggregate_performance.sort_values(["target", "mean_rmse"]))
display(ranking)

## English Table 1 source and historical reproducibility audit

In [ ]:
table1 = aggregate_performance.pivot(
    index=["family", "family_label"], columns="target", values=["mean_rmse", "mean_r2"]
).reset_index()
table1.columns = [
    "family", "Family",
    "Relative humidity RMSE (p.p.)", "Temperature RMSE (°C)",
    "Relative humidity R²", "Temperature R²",
]
table1 = table1[[
    "family", "Family", "Temperature RMSE (°C)", "Temperature R²",
    "Relative humidity RMSE (p.p.)", "Relative humidity R²",
]].sort_values("Temperature RMSE (°C)")
table1.to_csv(RESULTS_DIR / "09_table1_aggregate_predictive_performance.csv", index=False)

HISTORICAL_TABLE1 = {
    "HYBRID_ROBUST": {"temperature": (1.3329348598850272, 0.8956021532007421), "relative_humidity": (3.182132976915415, 0.9263202251644789)},
    "ADVANCED_TRADITIONAL": {"temperature": (1.3381581395598836, 0.8948595723226991), "relative_humidity": (3.180356958801399, 0.9261529683592873)},
    "MACHINE_LEARNING": {"temperature": (1.3455278668484731, 0.8922208251301068), "relative_humidity": (3.5427171416868104, 0.908055281581133)},
    "STATISTICAL": {"temperature": (1.37750739551782, 0.8869046964966601), "relative_humidity": (3.6129710800279176, 0.9045657789828958)},
    "DEEP_LEARNING": {"temperature": (1.6004680647777252, 0.8519134081171824), "relative_humidity": (4.274297471748146, 0.8687114821190555)},
    "BASELINE": {"temperature": (1.7472535580589774, 0.826286270021639), "relative_humidity": (4.502806511295871, 0.8591284160692101)},
}
historical_rows = []
for row in aggregate_performance.itertuples(index=False):
    expected_rmse, expected_r2 = HISTORICAL_TABLE1[row.family][row.target]
    historical_rows.append({
        "family": row.family,
        "target": row.target,
        "observed_mean_rmse": row.mean_rmse,
        "historical_mean_rmse": expected_rmse,
        "rmse_difference": row.mean_rmse - expected_rmse,
        "observed_mean_r2": row.mean_r2,
        "historical_mean_r2": expected_r2,
        "r2_difference": row.mean_r2 - expected_r2,
        "within_0_001": abs(row.mean_rmse - expected_rmse) <= 0.001 and abs(row.mean_r2 - expected_r2) <= 0.001,
    })
historical_audit = pd.DataFrame(historical_rows)
historical_audit.to_csv(RESULTS_DIR / "10_historical_table1_audit.csv", index=False)
display(table1.drop(columns="family").round(4))
display(historical_audit)

## Selection manifest, validation report, and metadata

In [ ]:
selection_manifest = pd.DataFrame([
    {"family": "BASELINE", "source_notebook": "04", "selection_scope": "task", "selection_basis": "validation RMSE", "test_used_for_selection": False},
    {"family": "STATISTICAL", "source_notebook": "04", "selection_scope": "task", "selection_basis": "validation RMSE", "test_used_for_selection": False},
    {"family": "MACHINE_LEARNING", "source_notebook": "05", "selection_scope": "resolution", "selection_basis": "validation NRMSE/RMSE and R2", "test_used_for_selection": False},
    {"family": "DEEP_LEARNING", "source_notebook": "06 + notebook 09 operational selector", "selection_scope": "task", "selection_basis": "validation ensemble RMSE + seed stability", "test_used_for_selection": False},
    {"family": "ADVANCED_TRADITIONAL", "source_notebook": "07", "selection_scope": "resolution × target", "selection_basis": "mean validation NRMSE across horizons", "test_used_for_selection": False},
    {"family": "HYBRID_ROBUST", "source_notebook": "08", "selection_scope": "task", "selection_basis": "validation-only multi-window robustness", "test_used_for_selection": False},
])
selection_manifest.to_csv(RESULTS_DIR / "02_family_selection_manifest.csv", index=False)

final_predictions = final_predictions.sort_values(["family"] + KEY_COLUMNS).reset_index(drop=True)
final_predictions.to_parquet(PREDICTION_DIR / "01_family_selected_test_predictions.parquet", index=False)

validation_rows = [
    {"check": "six_families_present", "passed": set(final_predictions["family"]) == set(FAMILIES), "detail": sorted(final_predictions["family"].unique())},
    {"check": "expected_tasks_per_family", "passed": bool((family_task_audit["tasks"] == EXPECTED_TASKS).all()), "detail": family_task_audit.set_index("family")["tasks"].to_dict()},
    {"check": "origin_key_parity", "passed": bool(origin_parity_audit["key_match_to_canonical"].all()), "detail": "canonical=ADVANCED_TRADITIONAL"},
    {"check": "observed_value_parity", "passed": bool(origin_parity_audit["truth_match_to_canonical"].all()), "detail": f"atol={TRUTH_ATOL}, rtol={TRUTH_RTOL}"},
    {"check": "base_only_exact_equivalence", "passed": bool(base_only_summary["exact_equivalence"].iloc[0]), "detail": base_only_summary.to_dict("records")[0]},
    {"check": "finite_metrics", "passed": bool(task_metrics[["rmse", "nrmse", "mae", "r2"]].apply(np.isfinite).all().all()), "detail": f"rows={len(task_metrics)}"},
    {"check": "no_test_selection", "passed": bool((~selection_manifest["test_used_for_selection"]).all()), "detail": "all family selectors are validation-only"},
]
validation_report = pd.DataFrame(validation_rows)
validation_report["detail"] = validation_report["detail"].map(str)
assert validation_report["passed"].all(), validation_report
validation_report.to_csv(RESULTS_DIR / "11_output_validation.csv", index=False)

configuration = {
    "notebook": "09_final_model_family_benchmark.ipynb",
    "execution_mode": EXECUTION_MODE,
    "resolutions_minutes": ACTIVE_RESOLUTIONS,
    "targets": TARGETS,
    "horizons_minutes": HORIZONS,
    "families": FAMILIES,
    "expected_tasks_per_family": EXPECTED_TASKS,
    "canonical_truth_source": "ADVANCED_TRADITIONAL",
    "deep_learning_candidate_id_source": deep_learning_candidate_id_source,
    "normalization": "test-task target standard deviation, ddof=0",
    "truth_alignment_tolerances": {"absolute": TRUTH_ATOL, "relative": TRUTH_RTOL},
    "historical_values_are_audit_only": True,
    "test_used_for_selection": False,
}
(METADATA_DIR / "09_final_benchmark_configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)
runtime = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "project_root": str(PROJECT_ROOT),
}
(METADATA_DIR / "09_final_benchmark_runtime.json").write_text(
    json.dumps(runtime, indent=2), encoding="utf-8"
)

display(selection_manifest)
display(validation_report)

## Completion criteria

In a full run, the notebook must produce:

- 6 model families;
- 40 tasks per family and 240 task-level metric rows;
- identical test-origin keys and observed values across all families;
- exact equality between every robust `BASE_ONLY` prediction and its advanced-traditional counterpart;
- the English Table 1 source in `results/final_benchmark/09_table1_aggregate_predictive_performance.csv`;
- the frozen prediction panel for notebook `10_statistical_analysis_and_figures.ipynb`.

The historical Table 1 comparison is an audit, not an assertion. Version-related numerical differences must be investigated and documented, never overwritten with historical values.